# Interactive Multi-Tool ReAct Agent with NVIDIA Nemotron on Gemini Enterprise Agent Platform

> **Google Cloud | Gemini Enterprise Agent Platform | NVIDIA Nemotron in Model Garden**

---

### 📖 Executive Overview
This notebook is your interactive starting point for building **Autonomous Multi-Tool ReAct (Reasoning + Acting) Agents** using the **NVIDIA Nemotron** foundation model family on **Google Cloud Gemini Enterprise Agent Platform**.

Whether your target is **Nemotron-3.5 Lightning**, **Nemotron-3 Nano (30B/3B)**, **Nemotron-3 Super (120B/12B)**, or **Nemotron-3 Ultra (550B/55B)**, this notebook supports:
1. **Auto-Discovery**: Seamlessly connects to active Model Garden deployments in your GCP project.
2. **Custom Endpoint Integration**: Bind to any existing custom endpoint, fine-tuned model, or private endpoint.
3. **Programmatic Endpoint Creation**: Create and deploy a new custom Vertex AI endpoint directly from the notebook.
4. **ReAct Autonomous Cycle**: Step-by-step reasoning (*Thought* $\rightarrow$ *Action* $\rightarrow$ *Observation* $\rightarrow$ *Final Answer*).
5. **Enterprise Tool Suite**: Live Python REPL calculator, Google Cloud infrastructure metadata lookups, and cloud architectural knowledge retrieval.

> [!TIP]
> **Flexible Endpoint Operation Modes:**
> * **Mode 1 (`AUTO_DISCOVER`) [Default]**: Automatically scans and binds to active Model Garden deployments in your GCP project. (To deploy via UI ahead of time: [Vertex AI Model Garden](https://console.cloud.google.com/vertex-ai/model-garden), recommended profile: **g4-standard-48 or g2-standard-16**).
> * **Mode 2 (`USE_EXISTING_ENDPOINT`)**: Bind directly to any custom endpoint or fine-tuned model by setting `CUSTOM_ENDPOINT_ID`.
> * **Mode 3 (`CREATE_CUSTOM_ENDPOINT`)**: Programmatically create a new Vertex AI Endpoint and deploy your custom container or model weights directly from the notebook.

---

### 📋 Prerequisites & Setup
* **Google Cloud Project**: With Vertex AI / Gemini Enterprise Agent Platform API enabled (`aiplatform.googleapis.com`).
* **Authentication**: Run `gcloud auth application-default login` if running locally or in Cloud Shell.
* **Compute / Accelerator**: A Model Garden Nemotron endpoint or custom endpoint deployed on NVIDIA GPUs (e.g. `g4-standard-48`, `g4-standard-384`, or `g2-standard-16`).


### Step 1: Harmonized Dependency Installation
Run the cell below to install required client libraries. Version constraints are pre-harmonized to eliminate dependency conflicts with Google Colab and Vertex AI Workbench.


In [ ]:
# Copyright 2026 Google LLC
#
# Licensed under the Apache License, Version 2.0 (the "License");
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
#     https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

import sys
import subprocess

# Harmonized dependency installation with conflict prevention
!pip install --quiet --no-warn-conflicts "google-cloud-aiplatform>=1.70.0" "openai>=1.50.0,<2.0.0" "pydantic>=2.0.0,<3.0.0" "rich>=13.7.0,<14.0.0" "requests>=2.31.0,<=2.32.4" "protobuf>=3.20.2,<5.0.0dev"

print("✓ Harmonized dependencies installed successfully.")


### Step 2: Environment Configuration & Endpoint Operation Selector
Configure your Google Cloud Project settings and select your endpoint connection mode.

> [!TIP]
> **Flexible Endpoint Operation Modes:**
> * **Mode 1 (`AUTO_DISCOVER`) [Default]**: Automatically scans and binds to active Model Garden deployments in your GCP project. (To deploy via UI ahead of time: [Vertex AI Model Garden](https://console.cloud.google.com/vertex-ai/model-garden), recommended profile: **g4-standard-48 or g2-standard-16**).
> * **Mode 2 (`USE_EXISTING_ENDPOINT`)**: Bind directly to any custom endpoint or fine-tuned model by setting `CUSTOM_ENDPOINT_ID`.
> * **Mode 3 (`CREATE_CUSTOM_ENDPOINT`)**: Programmatically create a new Vertex AI Endpoint and deploy your custom container or model weights directly from the notebook.


In [ ]:
import os
import subprocess
from google.cloud import aiplatform
from rich.console import Console
from rich.table import Table
from rich.panel import Panel

console = Console()

# ==============================================================================
# Step 2: Environment Configuration & Endpoint Operation Selector
# ==============================================================================
# Choose your connection/deployment operation:
# 1. "AUTO_DISCOVER" : (Default) Auto-detects and binds to active Model Garden endpoints.
# 2. "USE_EXISTING_ENDPOINT": Binds directly to your custom or existing endpoint ID/Name.
# 3. "CREATE_CUSTOM_ENDPOINT": Programmatically creates a new Vertex AI endpoint and
#                              deploys a custom container/model artifact.
# ==============================================================================

PROJECT_ID = ""                # @param {type:"string"} - Set your GCP Project ID (Leave blank to auto-detect)
REGION = "us-central1"         # @param ["us-central1", "us-east4", "us-west1", "europe-west4"] {allow-input: true}
OPERATION_MODE = "AUTO_DISCOVER" # @param ["AUTO_DISCOVER", "USE_EXISTING_ENDPOINT", "CREATE_CUSTOM_ENDPOINT"]

# --- Mode: USE_EXISTING_ENDPOINT Settings ---
CUSTOM_ENDPOINT_ID = ""        # @param {type:"string"} - e.g. "1234567890" or "projects/.../endpoints/..."

# --- Mode: CREATE_CUSTOM_ENDPOINT Settings ---
CUSTOM_ENDPOINT_DISPLAY_NAME = "nemotron-custom-endpoint" # @param {type:"string"}
CUSTOM_SERVING_CONTAINER_URI = "us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/vllm-serve:latest" # @param {type:"string"}
CUSTOM_ARTIFACT_URI = ""       # @param {type:"string"} - Optional: Cloud Storage path (e.g. gs://your-bucket/model-weights)
CUSTOM_MACHINE_TYPE = "g4-standard-48" # @param ["g4-standard-48", "g4-standard-384", "g2-standard-16", "g2-standard-96", "a4-highgpu-8g", "a3-ultragpu-8g"] {allow-input: true}
CUSTOM_ACCELERATOR_TYPE = "NVIDIA_RTX_PRO_6000" # @param ["NVIDIA_RTX_PRO_6000", "NVIDIA_L4", "NVIDIA_B200", "NVIDIA_H200"] {allow-input: true}
CUSTOM_ACCELERATOR_COUNT = 1   # @param {type:"integer"}

# 1. Resolve GCP Project ID
if not PROJECT_ID.strip():
    try:
        PROJECT_ID = subprocess.check_output(
            ["gcloud", "config", "get-value", "project"], 
            stderr=subprocess.DEVNULL
        ).decode().strip()
    except Exception:
        PROJECT_ID = os.environ.get("GOOGLE_CLOUD_PROJECT", "cpe-slarbi-nvd-ant-demos")

console.print(f"[bold green]✓ GCP Project:[/bold green] [cyan]{PROJECT_ID}[/cyan] | [bold green]Region:[/bold green] [cyan]{REGION}[/cyan] | [bold green]Mode:[/bold green] [yellow]{OPERATION_MODE}[/yellow]")
aiplatform.init(project=PROJECT_ID, location=REGION)

# 2. Unified Endpoint Resolver & Deployer
def resolve_or_create_endpoint(
    project_id: str, 
    location: str, 
    target_keywords: list,
    mode: str = "AUTO_DISCOVER",
    custom_endpoint_id: str = "",
    create_params: dict = None
) -> aiplatform.Endpoint:
    # -------------------------------------------------------------------------
    # Path 1: Connect to an Existing Custom Endpoint
    # -------------------------------------------------------------------------
    if mode == "USE_EXISTING_ENDPOINT" or (custom_endpoint_id and custom_endpoint_id.strip()):
        ep_name = custom_endpoint_id.strip()
        if not ep_name:
            raise ValueError("OPERATION_MODE is 'USE_EXISTING_ENDPOINT', but CUSTOM_ENDPOINT_ID is empty.")
        
        ep_path = ep_name if ep_name.startswith("projects/") else f"projects/{project_id}/locations/{location}/endpoints/{ep_name}"
        ep = aiplatform.Endpoint(ep_path)
        console.print(Panel(
            f"[bold]Display Name:[/bold] {ep.display_name}\n"
            f"[bold]Endpoint ID:[/bold]  {ep.name.split('/')[-1]}\n"
            f"[bold]Resource:[/bold]     {ep.name}",
            title="✓ BOUND TO CUSTOM ENDPOINT",
            border_style="green"
        ))
        return ep

    # -------------------------------------------------------------------------
    # Path 2: Create & Deploy a Custom Endpoint on Demand
    # -------------------------------------------------------------------------
    if mode == "CREATE_CUSTOM_ENDPOINT":
        p = create_params or {}
        disp_name = p.get("display_name", "custom-nemotron-endpoint")
        container_uri = p.get("container_uri", "us-docker.pkg.dev/vertex-ai/vertex-vision-model-garden-dockers/vllm-serve:latest")
        artifact_uri = p.get("artifact_uri", "")
        mach_type = p.get("machine_type", "g4-standard-48")
        acc_t = p.get("accelerator_type", "NVIDIA_RTX_PRO_6000")
        acc_c = p.get("accelerator_count", 1)

        console.print(Panel(
            f"[bold]Endpoint Name:[/bold]     {disp_name}\n"
            f"[bold]Serving Container:[/bold] {container_uri}\n"
            f"[bold]Hardware Profile:[/bold]  {mach_type} ({acc_c}x {acc_t})",
            title="🚀 INITIATING CUSTOM ENDPOINT DEPLOYMENT",
            border_style="yellow"
        ))

        console.print("⏳ [1/3] Registering custom model with Vertex AI...")
        upload_kwargs = {
            "display_name": f"{disp_name}-model",
            "serving_container_image_uri": container_uri,
        }
        if artifact_uri.strip():
            upload_kwargs["artifact_uri"] = artifact_uri.strip()

        model_res = aiplatform.Model.upload(**upload_kwargs)
        console.print(f"✓ Model registered: [cyan]{model_res.resource_name}[/cyan]")

        console.print("⏳ [2/3] Creating dedicated Vertex AI endpoint...")
        custom_endpoint = aiplatform.Endpoint.create(display_name=disp_name)
        console.print(f"✓ Endpoint created: [cyan]{custom_endpoint.resource_name}[/cyan]")

        console.print("⏳ [3/3] Deploying model to endpoint (provisioning compute and loading weights)...")
        model_res.deploy(
            endpoint=custom_endpoint,
            machine_type=mach_type,
            accelerator_type=acc_t,
            accelerator_count=acc_c,
            traffic_percentage=100,
            sync=True
        )
        console.print(Panel(
            f"[bold]Display Name:[/bold] {custom_endpoint.display_name}\n"
            f"[bold]Endpoint ID:[/bold]  {custom_endpoint.name.split('/')[-1]}\n"
            f"[bold]Resource:[/bold]     {custom_endpoint.name}",
            title="✓ CUSTOM ENDPOINT DEPLOYED SUCCESSFULLY",
            border_style="bold green"
        ))
        return custom_endpoint

    # -------------------------------------------------------------------------
    # Path 3: Auto-Discovery of Active Model Garden Endpoints (Default)
    # -------------------------------------------------------------------------
    console.print(f"🔍 Scanning for active Vertex AI endpoints in [cyan]{project_id}[/cyan] ({location})...")
    endpoints = aiplatform.Endpoint.list(order_by="create_time desc")
    
    if not endpoints:
        console.print(Panel(
            f"No active endpoints found in project [cyan]{project_id}[/cyan] / [cyan]{location}[/cyan].\n\n"
            f"1. Open Model Garden: https://console.cloud.google.com/vertex-ai/model-garden\n"
            f"2. Or set OPERATION_MODE = 'CREATE_CUSTOM_ENDPOINT' to deploy directly.\n"
            f"3. Or set OPERATION_MODE = 'USE_EXISTING_ENDPOINT' with CUSTOM_ENDPOINT_ID.",
            title="⚠️ NO ACTIVE ENDPOINTS FOUND",
            border_style="bold red"
        ))
        raise RuntimeError("No active endpoints found. Please deploy a Model Garden model or select a custom mode.")

    table = Table(title=f"Active Endpoints in {project_id}", border_style="blue")
    table.add_column("#", style="dim", width=4)
    table.add_column("Display Name", style="bold white")
    table.add_column("Endpoint ID", style="cyan")
    
    for idx, ep in enumerate(endpoints, start=1):
        table.add_row(str(idx), ep.display_name, ep.name.split("/")[-1])
    console.print(table)

    # 1st Priority: Match target model keywords
    matching = [
        ep for ep in endpoints 
        if any(k.lower() in (ep.display_name or "").lower() for k in target_keywords)
    ]

    if matching:
        selected = matching[0]
        console.print(Panel(
            f"[bold]Attached Model:[/bold]   {selected.display_name}\n"
            f"[bold]Endpoint ID:[/bold]      {selected.name.split('/')[-1]}\n"
            f"[bold]Resource Path:[/bold]    {selected.name}",
            title=f"✓ AUTO-ATTACHED: NVIDIA Nemotron",
            border_style="bold green"
        ))
        return selected

    # 2nd Priority: Fallback to any active Nemotron / NVIDIA endpoint
    generic_nemotron = [
        ep for ep in endpoints 
        if any(k in (ep.display_name or "").lower() for k in ["nemotron", "nvidia"])
    ]
    if generic_nemotron:
        selected = generic_nemotron[0]
        console.print(Panel(
            f"[bold]Attached Model:[/bold]   {selected.display_name}\n"
            f"[bold]Endpoint ID:[/bold]      {selected.name.split('/')[-1]}\n"
            f"[bold]Resource Path:[/bold]    {selected.name}",
            title="💡 AUTO-ATTACHED TO ACTIVE NEMOTRON ENDPOINT",
            border_style="bold yellow"
        ))
        return selected

    # 3rd Priority: Fallback to first available active endpoint
    selected = endpoints[0]
    console.print(f"💡 [dim]Fallback: Binding to active endpoint '{selected.display_name}' (ID: {selected.name.split('/')[-1]}).[/dim]")
    return selected

target_endpoint = resolve_or_create_endpoint(
    PROJECT_ID, 
    REGION, 
    target_keywords=["nemotron", "nvidia", "super", "nano", "lightning", "ultra"],
    mode=OPERATION_MODE,
    custom_endpoint_id=CUSTOM_ENDPOINT_ID,
    create_params={
        "display_name": CUSTOM_ENDPOINT_DISPLAY_NAME,
        "container_uri": CUSTOM_SERVING_CONTAINER_URI,
        "artifact_uri": CUSTOM_ARTIFACT_URI,
        "machine_type": CUSTOM_MACHINE_TYPE,
        "accelerator_type": CUSTOM_ACCELERATOR_TYPE,
        "accelerator_count": CUSTOM_ACCELERATOR_COUNT
    }
)


### Step 3: Model Serving Engine & Inference Client
Wraps prediction calls with generous token limits (`max_tokens=1024`) and robust response parsing to extract clean content from native Vertex AI and OpenAI-compatible `/v1/chat/completions` formats.


In [ ]:
import json
import re

def extract_clean_content(prediction_obj) -> str:
    """
    Extracts purely the clean assistant text from any Vertex AI / vLLM / OpenAI response format,
    filtering out raw API dictionaries, metadata envelopes, usage stats, and unicode artifacts.
    """
    if isinstance(prediction_obj, list) and len(prediction_obj) > 0:
        prediction_obj = prediction_obj[0]

    if isinstance(prediction_obj, str):
        try:
            prediction_obj = json.loads(prediction_obj)
        except Exception:
            pass

    content = ""
    if isinstance(prediction_obj, dict):
        # 1. Check for OpenAI/vLLM 'choices' format
        choices = prediction_obj.get("choices", [])
        if isinstance(choices, list) and len(choices) > 0:
            first_choice = choices[0]
            if isinstance(first_choice, dict):
                msg = first_choice.get("message", {})
                if isinstance(msg, dict):
                    content = msg.get("content", "")
                    if not content and "reasoning" in msg:
                        content = msg.get("reasoning", "")
                    if not content and "reasoning_content" in msg:
                        content = msg.get("reasoning_content", "")
                elif isinstance(msg, str):
                    content = msg
                if not content:
                    content = first_choice.get("text", "")
            elif isinstance(first_choice, str):
                content = first_choice

        # 2. Check for standard 'content', 'text', or 'predictions'
        if not content:
            content = prediction_obj.get("content", prediction_obj.get("text", ""))

        # 3. Direct message dict check
        if not content and "role" in prediction_obj and "content" in prediction_obj:
            content = prediction_obj.get("content", "")

    elif isinstance(prediction_obj, str):
        content = prediction_obj

    if not content:
        content = str(prediction_obj)

    # Normalize unicode spacing artifacts ( ,  )
    cleaned = str(content).replace("\u202f", " ").replace("\u00a0", " ").strip()
    return cleaned

def extract_json_from_text(raw_text: str) -> dict:
    """
    Robustly extracts and parses a JSON dictionary from LLM output,
    handling markdown blocks, preambles, reasoning wrappers, and trailing commas.
    """
    text = raw_text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass

    if "```json" in text:
        content = text.split("```json", 1)[1]
        if "```" in content:
            content = content.split("```", 1)[0]
        try:
            return json.loads(content.strip())
        except Exception:
            sanitized = re.sub(r',\s*([\}\]])', r'\1', content.strip())
            try:
                return json.loads(sanitized)
            except Exception:
                pass

    if "```" in text:
        content = text.split("```", 1)[1]
        if "```" in content:
            content = content.split("```", 1)[0]
        try:
            return json.loads(content.strip())
        except Exception:
            sanitized = re.sub(r',\s*([\}\]])', r'\1', content.strip())
            try:
                return json.loads(sanitized)
            except Exception:
                pass

    first_brace = text.find("{")
    last_brace = text.rfind("}")
    if first_brace != -1 and last_brace != -1 and last_brace > first_brace:
        candidate = text[first_brace:last_brace + 1]
        try:
            return json.loads(candidate)
        except Exception:
            sanitized = re.sub(r',\s*([\}\]])', r'\1', candidate)
            try:
                return json.loads(sanitized)
            except Exception:
                pass

    raise ValueError(f"No valid JSON object found in model output: {text[:200]}")

class NemotronClient:
    """Unified inference client for Model Garden and custom Nemotron endpoints."""
    def __init__(self, endpoint: aiplatform.Endpoint):
        self.endpoint = endpoint

    def generate(self, prompt: str, max_tokens: int = 1024, temperature: float = 0.2) -> str:
        payload = {
            "instances": [
                {
                    "@requestFormat": "chatCompletions",
                    "messages": [{"role": "user", "content": prompt}],
                    "max_tokens": max_tokens,
                    "temperature": temperature
                }
            ]
        }
        try:
            response = self.endpoint.predict(instances=payload["instances"])
            return extract_clean_content(response.predictions)
        except Exception:
            try:
                res = self.endpoint.predict(instances=[{"prompt": prompt, "max_tokens": max_tokens}])
                return extract_clean_content(res.predictions)
            except Exception as e:
                return f"[Inference Error: {str(e)}]"

client = NemotronClient(target_endpoint)
console.print("[bold green]✓ Nemotron Inference Client connected and ready.[/bold green]")


### Step 4: Register Production Agent Tools
We register 3 enterprise tools for the agent:
1. **`calculate_or_execute_python`**: Runs arithmetic and data manipulation in a sandbox.
2. **`query_gcp_resources`**: Retrieves hardware specs for Google Cloud G4 Blackwell and L4 machine types.
3. **`search_cloud_knowledge_base`**: Retrieves architecture best practices and latency guidelines.


In [ ]:
import io
import sys
import contextlib

def calculate_or_execute_python(code_str: str) -> str:
    """Executes a Python snippet safely and captures standard output for math/calculations."""
    buf = io.StringIO()
    try:
        with contextlib.redirect_stdout(buf):
            allowed_globals = {"__builtins__": __builtins__, "math": __import__("math")}
            exec(code_str, allowed_globals, {})
        output = buf.getvalue().strip()
        return output if output else "[Executed successfully with no stdout]"
    except Exception as e:
        return f"Execution Error: {str(e)}"

def query_gcp_resources(machine_type: str) -> str:
    """Queries hardware specifications for Google Cloud accelerator profiles."""
    db = {
        "g4-standard-48": "Machine Type: g4-standard-48 | GPU: 1x NVIDIA RTX PRO 6000 (Blackwell 48GB) | vCPU: 48 | RAM: 192GB | Max TFLOPS: 1200",
        "g4-standard-96": "Machine Type: g4-standard-96 | GPU: 2x NVIDIA RTX PRO 6000 (Blackwell 96GB) | vCPU: 96 | RAM: 384GB",
        "g4-standard-384": "Machine Type: g4-standard-384 | GPU: 8x NVIDIA RTX PRO 6000 (Blackwell 384GB) | vCPU: 384 | RAM: 1536GB",
        "g2-standard-16": "Machine Type: g2-standard-16 | GPU: 1x NVIDIA L4 (24GB VRAM) | vCPU: 16 | RAM: 64GB",
        "g2-standard-96": "Machine Type: g2-standard-96 | GPU: 8x NVIDIA L4 (192GB VRAM) | vCPU: 96 | RAM: 384GB",
        "a4-highgpu-8g": "Machine Type: a4-highgpu-8g | GPU: 8x NVIDIA B200 (1536GB VRAM) | vCPU: 224 | RAM: 1800GB",
        "a3-ultragpu-8g": "Machine Type: a3-ultragpu-8g | GPU: 8x NVIDIA H200 (1128GB VRAM) | vCPU: 208 | RAM: 1872GB"
    }
    key = machine_type.strip().lower()
    for k, v in db.items():
        if k in key:
            return v
    return f"Available profiles in catalog: {list(db.keys())}"

def search_cloud_knowledge_base(query: str) -> str:
    """Retrieves Google Cloud architectural guidelines and latency recommendations."""
    query_lower = query.lower()
    if "nemotron" in query_lower or "moe" in query_lower or "latency" in query_lower:
        return ("Best Practice: Nemotron-3 MoE uses sparse routing (e.g. 55B active of 550B total). "
                "For ultra-low latency SLAs (< 500ms TTFT), use Nemotron-3.5 Lightning on G4 Blackwell. "
                "For 256K long-context compliance pipelines, deploy Nemotron-3 Ultra with NVFP4 quantization.")
    elif "cost" in query_lower or "sizing" in query_lower:
        return "Sizing Guide: G4 instances provide 2.5x throughput per dollar compared to prior generation Hopper instances on FP4 workloads."
    return "Knowledge Base: Refer to Google Cloud Model Garden & Gemini Enterprise Agent Platform architecture guides."

TOOL_REGISTRY = {
    "calculate_or_execute_python": calculate_or_execute_python,
    "query_gcp_resources": query_gcp_resources,
    "search_cloud_knowledge_base": search_cloud_knowledge_base
}

tool_table = Table(title="Registered Enterprise Tools", border_style="cyan")
tool_table.add_column("Tool Name", style="bold green")
tool_table.add_column("Description", style="white")
for t_name, t_fn in TOOL_REGISTRY.items():
    tool_table.add_row(t_name, t_fn.__doc__.strip().splitlines()[0])
console.print(tool_table)


### Step 5: The Autonomous ReAct Execution Engine (with Clean Formatted Traces & Max Tokens)
Implements the multi-turn Thought $\rightarrow$ Action $\rightarrow$ Observation loop with structured visual formatting and generous token budgets (`max_tokens=1024`) to accommodate deep reasoning.


In [ ]:
import re
from IPython.display import display, Markdown

REACT_SYSTEM_PROMPT = """You are an autonomous Cloud Architect powered by NVIDIA Nemotron on Google Cloud Gemini Enterprise Agent Platform.
You solve complex engineering challenges using the ReAct (Reason + Act) paradigm.

You have access to the following tools:
1. calculate_or_execute_python[code]: Runs python code to calculate exact mathematical, sizing, or financial numbers.
2. query_gcp_resources[machine_type]: Retrieves specifications for Google Cloud accelerator profiles (e.g. g4-standard-48, g4-standard-384, a4-highgpu-8g).
3. search_cloud_knowledge_base[query]: Retrieves architectural best practices and latency recommendations.

Use the following format:
Question: the input question you must answer
Thought: reason about what step to take next
Action: the action to take, exactly matching `tool_name[argument]`
Observation: the result of the action
... (this Thought/Action/Observation can repeat up to 5 times)
Thought: I now have enough information to answer the question
Final Answer: the comprehensive final answer to the original question.

Begin!"""

def render_agent_output(title: str, content: str):
    """Renders clean, styled Markdown inside Jupyter notebooks."""
    try:
        display(Markdown(f"### {title}\n\n{content}"))
    except Exception:
        console.print(Panel(content, title=title, border_style="bold green"))

def run_react_agent(user_query: str, max_turns: int = 5) -> str:
    """Executes the ReAct loop with generous token allocations (1024 tokens/turn) and clean formatted traces."""
    prompt_history = f"{REACT_SYSTEM_PROMPT}\n\nQuestion: {user_query}\n"
    console.print(Panel(f"[bold white]{user_query}[/bold white]", title="🚀 USER QUESTION", border_style="bold cyan"))

    for turn in range(1, max_turns + 1):
        console.rule(f"[bold cyan]Turn {turn}/{max_turns}[/bold cyan]", style="cyan")
        response = client.generate(prompt_history, max_tokens=1024, temperature=0.1)
        
        # Cleanly display Thought
        thought_match = re.search(r"Thought:\s*(.+?)(?=\nAction:|\nFinal Answer:|$)", response, re.DOTALL)
        if thought_match:
            thought_text = thought_match.group(1).strip()
            console.print(f"[bold yellow]🤔 Thought:[/bold yellow] [italic]{thought_text}[/italic]")
        elif not ("Action:" in response or "Final Answer:" in response):
            console.print(f"[bold yellow]🤔 Thought:[/bold yellow] [italic]{response}[/italic]")

        if "Final Answer:" in response:
            final_ans = response.split("Final Answer:")[-1].strip()
            render_agent_output("🎯 FINAL AGENT ANSWER", final_ans)
            return final_ans

        action_match = re.search(r"Action:\s*([a-zA-Z_0-9]+)\[([^\]]+)\]", response)
        if not action_match:
            if turn == max_turns:
                render_agent_output("🎯 FINAL AGENT ANSWER", response.strip())
                return response.strip()
            prompt_history += f"\n{response}\nObservation: [Please proceed to next action or provide Final Answer]\n"
            continue

        tool_name = action_match.group(1).strip()
        tool_arg = action_match.group(2).strip()

        console.print(f"[bold magenta]⚙️ Action:[/bold magenta] [bold white]{tool_name}[/bold white]([cyan]{tool_arg}[/cyan])")

        if tool_name in TOOL_REGISTRY:
            tool_output = TOOL_REGISTRY[tool_name](tool_arg)
        else:
            tool_output = f"Error: Tool '{tool_name}' not recognized. Available tools: {list(TOOL_REGISTRY.keys())}"

        console.print(f"[bold green]📥 Observation:[/bold green] {tool_output}\n")
        prompt_history += f"\n{response}\nObservation: {tool_output}\n"

    final_wrap = client.generate(prompt_history + "\nThought: I will summarize the final answer now.\nFinal Answer:", max_tokens=1024)
    wrap_ans = final_wrap.split("Final Answer:")[-1].strip() if "Final Answer:" in final_wrap else final_wrap.strip()
    render_agent_output("🎯 FINAL AGENT ANSWER", wrap_ans)
    return wrap_ans


### Step 6: Live Enterprise Scenarios
Execute the cells below to observe the agent solving multi-step engineering tasks autonomously with clean Markdown rendering.


In [ ]:
# Scenario 1: Multi-Step Capacity Sizing & Cost Analysis
scenario_1 = (
    "We need to deploy a high-throughput Nemotron cluster on Google Cloud. "
    "First, query the specifications for machine type 'g4-standard-384'. "
    "Then, calculate how many total vCPUs and total GPU VRAM we will have across a 4-node cluster using python code. "
    "Finally, state the sizing summary."
)

ans1 = run_react_agent(scenario_1)


In [ ]:
# Scenario 2: Architectural Best Practice & Latency SLA Consultation
scenario_2 = (
    "Search the cloud knowledge base for architectural recommendations regarding Nemotron MoE models "
    "and ultra-low latency SLAs, then summarize the ideal deployment configuration."
)

ans2 = run_react_agent(scenario_2)


### Step 7: Interactive Query Tester
Test any custom prompt or engineering workflow using the interactive cell below.


In [ ]:
custom_query = "Query the specs for g4-standard-48 and calculate the memory per vCPU ratio in GB."
ans_custom = run_react_agent(custom_query)
